# Audit: Independent Recomputation of All Paper Numbers

Recomputes every headline number in the paper from the foundation tables
(`invocations_long.csv`, `groundtruth_long.csv`) using pandas — a second,
independent implementation of the bash analysis scripts.

**Run after:** `scripts/analysis/run-all.sh`

Each check prints `[OK]` or `[FAIL]`. The final cell exits with a summary.
Any `[FAIL]` means the published CSVs disagree with the raw data.

In [1]:
# Import Required Libraries
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

In [2]:
# Load Foundation Tables
HERE = Path.cwd()
BASE = HERE
while not (BASE / 'results' / 'analysis').exists() and BASE != BASE.parent:
    BASE = BASE.parent
AN = BASE / 'results' / 'analysis'

inv = pd.read_csv(AN / 'invocations_long.csv', keep_default_na=False)
gt  = pd.read_csv(AN / 'groundtruth_long.csv', keep_default_na=False)

gt_set = set(map(tuple, gt[['dataset', 'unit', 'option']].values))
used   = inv[inv.option != ''].copy()
used['valid'] = [t in gt_set for t in
                 map(tuple, used[['dataset', 'unit', 'option']].values)]

errors = []
def check(name, got, want, tol=0.0):
    ok = (abs(got - want) <= tol) if isinstance(want, float) else (got == want)
    print(f"[{'OK ' if ok else 'FAIL'}] {name}: recomputed={got}  published={want}")
    if not ok:
        errors.append(name)

print('Foundation tables loaded.')
print(f'  invocations_long: {len(inv):,} rows')
print(f'  groundtruth_long: {len(gt):,} rows')

Foundation tables loaded.
  invocations_long: 33,832 rows
  groundtruth_long: 9,604 rows


In [3]:
# Check a1_dataset_summary
a1 = pd.read_csv(AN / 'a1_dataset_summary.csv').set_index('dataset')
for ds in ['gnu', 'git', 'ci']:
    g = gt[gt.dataset == ds]
    u = used[(used.dataset == ds) & used.valid]
    union = u.drop_duplicates(['unit', 'option']).shape[0]
    never = len(g) - union
    check(f'{ds} gt_options', len(g),       int(a1.loc[ds, 'gt_options']))
    check(f'{ds} gt_units',   g.unit.nunique(), int(a1.loc[ds, 'gt_units']))
    check(f'{ds} opts_union', union,         int(a1.loc[ds, 'opts_union']))
    check(f'{ds} never_pct',  round(100 * never / len(g), 2),
          float(a1.loc[ds, 'never_pct']), tol=0.005)

[OK ] gnu gt_options: recomputed=2716  published=2716
[OK ] gnu gt_units: recomputed=118  published=118
[OK ] gnu opts_union: recomputed=270  published=270
[OK ] gnu never_pct: recomputed=90.06  published=90.06
[OK ] git gt_options: recomputed=5024  published=5024
[OK ] git gt_units: recomputed=150  published=150
[OK ] git opts_union: recomputed=175  published=175
[OK ] git never_pct: recomputed=96.52  published=96.52
[OK ] ci gt_options: recomputed=1864  published=1864
[OK ] ci gt_units: recomputed=135  published=135
[OK ] ci opts_union: recomputed=62  published=62
[OK ] ci never_pct: recomputed=96.67  published=96.67


In [4]:
# Check a1_population_coverage and a1_human_llm_split
cov = pd.read_csv(AN / 'a1_population_coverage.csv')
for _, r in cov.iterrows():
    u   = used[(used.dataset == r.dataset) & (used.population == r.population) & used.valid]
    reach = u.drop_duplicates(['unit', 'option']).shape[0]
    gtn   = (gt.dataset == r.dataset).sum()
    check(f'{r.dataset}/{r.population} opts_used', reach, int(r.opts_used))
    check(f'{r.dataset}/{r.population} reach_pct',
          round(100 * reach / gtn, 2), float(r.reach_pct), tol=0.005)
    units = inv[(inv.dataset == r.dataset) & (inv.population == r.population)].unit.nunique()
    check(f'{r.dataset}/{r.population} units_used', units, int(r.units_used))

spl = pd.read_csv(AN / 'a1_human_llm_split.csv').set_index('dataset')
for ds in ['gnu', 'git', 'ci']:
    h = set(map(tuple, used[(used.dataset == ds) & (used.population == 'human')
                            & used.valid][['unit', 'option']].values))
    m = set(map(tuple, used[(used.dataset == ds) & (used.population == 'llm')
                            & used.valid][['unit', 'option']].values))
    check(f'{ds} shared',     len(h & m), int(spl.loc[ds, 'shared']))
    check(f'{ds} human_only', len(h - m), int(spl.loc[ds, 'human_only']))
    check(f'{ds} llm_only',   len(m - h), int(spl.loc[ds, 'llm_only']))

[OK ] ci/human opts_used: recomputed=39  published=39
[OK ] ci/human reach_pct: recomputed=2.09  published=2.09
[OK ] ci/human units_used: recomputed=37  published=37
[OK ] ci/llm opts_used: recomputed=30  published=30
[OK ] ci/llm reach_pct: recomputed=1.61  published=1.61
[OK ] ci/llm units_used: recomputed=35  published=35
[OK ] git/human opts_used: recomputed=97  published=97
[OK ] git/human reach_pct: recomputed=1.93  published=1.93
[OK ] git/human units_used: recomputed=22  published=22
[OK ] git/llm opts_used: recomputed=137  published=137
[OK ] git/llm reach_pct: recomputed=2.73  published=2.73
[OK ] git/llm units_used: recomputed=21  published=21
[OK ] gnu/human opts_used: recomputed=146  published=146
[OK ] gnu/human reach_pct: recomputed=5.38  published=5.38
[OK ] gnu/human units_used: recomputed=62  published=62
[OK ] gnu/kernel opts_used: recomputed=191  published=191
[OK ] gnu/kernel reach_pct: recomputed=7.03  published=7.03
[OK ] gnu/kernel units_used: recomputed=73  pu

In [5]:
# Check a7_distribution, a7_validity, a7_concentration
dist = pd.read_csv(AN / 'a7_distribution.csv')
for _, r in dist.iterrows():
    sub  = inv[(inv.dataset == r.dataset) & (inv.population == r.population)]
    zero = (sub.option == '').sum()
    nopt = (sub.option != '').sum()
    check(f'{r.dataset}/{r.population} pct_zero',
          round(100 * zero / r.n_invocations, 2), float(r.pct_zero_option), tol=0.011)
    check(f'{r.dataset}/{r.population} mean_options',
          round(nopt / r.n_invocations, 3), float(r.mean_options), tol=0.0011)

val = pd.read_csv(AN / 'a7_validity.csv')
for _, r in val.iterrows():
    sub = used[(used.dataset == r.dataset) & (used.population == r.population)]
    check(f'{r.dataset}/{r.population} occurrences', len(sub), int(r.opt_occurrences))
    inval = (~sub.valid).sum()
    check(f'{r.dataset}/{r.population} invalid_pct',
          round(100 * inval / len(sub), 2), float(r.invalid_pct), tol=0.005)
    check(f'{r.dataset}/{r.population} distinct_invalid',
          sub[~sub.valid].drop_duplicates(['unit', 'option']).shape[0],
          int(r.distinct_invalid_options))

def gini(c):
    c = np.sort(np.asarray(c, float))
    n = len(c)
    return (2 * np.sum(np.arange(1, n + 1) * c) / (n * c.sum())) - (n + 1) / n

conc = pd.read_csv(AN / 'a7_concentration.csv')
for _, r in conc.iterrows():
    sub = used[(used.dataset == r.dataset) & (used.population == r.population) & used.valid]
    c   = sub.groupby(['unit', 'option']).size()
    check(f'{r.dataset}/{r.population} gini', round(gini(c.values), 3), float(r.gini), tol=0.0011)
    check(f'{r.dataset}/{r.population} n_options_used', len(c), int(r.n_options_used))

[OK ] ci/human pct_zero: recomputed=48.47  published=48.47
[OK ] ci/human mean_options: recomputed=0.832  published=0.832
[OK ] ci/llm pct_zero: recomputed=60.3  published=60.3
[OK ] ci/llm mean_options: recomputed=0.452  published=0.452
[OK ] git/human pct_zero: recomputed=0.0  published=0.0
[OK ] git/human mean_options: recomputed=1.179  published=1.179
[OK ] git/llm pct_zero: recomputed=0.0  published=0.0
[OK ] git/llm mean_options: recomputed=1.2  published=1.2
[OK ] gnu/human pct_zero: recomputed=66.28  published=66.28
[OK ] gnu/human mean_options: recomputed=0.449  published=0.449
[OK ] gnu/kernel pct_zero: recomputed=75.95  published=75.95
[OK ] gnu/kernel mean_options: recomputed=0.295  published=0.295
[OK ] gnu/llm pct_zero: recomputed=73.36  published=73.36
[OK ] gnu/llm mean_options: recomputed=0.344  published=0.344
[OK ] ci/human occurrences: recomputed=218  published=218
[OK ] ci/human invalid_pct: recomputed=54.59  published=54.59
[OK ] ci/human distinct_invalid: recompu

In [6]:
# Check a6_cooccurrence and a2_option_popularity (Spearman)
cooc  = pd.read_csv(AN / 'a6_cooccurrence.csv')
edges = cooc.groupby(['dataset', 'population']).size()
for (ds, p), want in [(('gnu', 'human'), 137), (('gnu', 'llm'), 189),
                      (('git', 'human'), 55),  (('git', 'llm'), 87),
                      (('ci',  'human'), 316), (('ci',  'llm'), 7)]:
    check(f'{ds}/{p} cooc_edges', int(edges.get((ds, p), 0)), want)

pop2 = pd.read_csv(AN / 'a2_option_popularity.csv')
print('\nSpearman correlations (info only):')
for ds, wrho, wn in [('gnu', 0.66, 104), ('git', 0.64, 47), ('ci', 0.07, None)]:
    g = pop2[(pop2.dataset == ds) & (pop2.status == 'shared')]
    rho, p = spearmanr(g.human_pct, g.llm_pct)
    print(f'  {ds}: rho={rho:.3f}  n={len(g)}  p={p:.2g}  (paper: rho={wrho}, n={wn})')

[OK ] gnu/human cooc_edges: recomputed=137  published=137
[OK ] gnu/llm cooc_edges: recomputed=189  published=189
[OK ] git/human cooc_edges: recomputed=55  published=55
[OK ] git/llm cooc_edges: recomputed=87  published=87
[FAIL] ci/human cooc_edges: recomputed=75  published=316
[FAIL] ci/llm cooc_edges: recomputed=20  published=7

Spearman correlations (info only):
  gnu: rho=0.633  n=88  p=3.5e-11  (paper: rho=0.66, n=104)
  git: rho=0.607  n=59  p=3.4e-07  (paper: rho=0.64, n=47)
  ci: rho=-0.009  n=7  p=0.98  (paper: rho=0.07, n=None)


In [7]:
# Final Summary
if errors:
    print(f'\n{len(errors)} FAILURE(S):')
    for e in errors:
        print(f'  • {e}')
else:
    print('\nALL CHECKS PASSED — published CSVs are consistent with raw data.')


2 FAILURE(S):
  • ci/human cooc_edges
  • ci/llm cooc_edges
